# Prediction Demo

Predicts the monthly rent of **new** listings with the final model (`models/final_model.joblib`, the chosen model
averaged over seeds 42, 43, 44) through `predict_listing` — the same function the app will call. It validates the
input, fills sensible defaults for optional fields (with a warning), and explains problems instead of crashing.

## 1. Setup

In [1]:
import sys, pathlib, warnings
ROOT = next((p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents) if (p / 'src' / 'config.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project folder (the one that contains src/).')
sys.path.insert(0, str(ROOT))   # project root -> `import src`, whatever the working folder
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)
from src.prediction import load_final, load_input_schema, predict_listing, InvalidListing

final, schema = load_final(), load_input_schema()   # run final_selection.ipynb first
print(final)

SeedAveragedModel(Gradient Boosting, seeds=[42, 43, 44])


## 2. Complete listings

In [2]:
listings = [
    {'City': 'Kolkata', 'Area Locality': 'Salt Lake City Sector 2', 'BHK': 2, 'Size': 900, 'Bathroom': 2,
     'current_floor': 3, 'total_floors': 5, 'Furnishing Status': 'Semi-Furnished', 'Area Type': 'Super Area',
     'Tenant Preferred': 'Bachelors/Family', 'Point of Contact': 'Contact Owner'},
    {'City': 'Mumbai', 'Area Locality': 'Andheri West', 'BHK': 3, 'Size': 1400, 'Bathroom': 3,
     'current_floor': 10, 'total_floors': 20, 'Furnishing Status': 'Furnished', 'Area Type': 'Carpet Area',
     'Tenant Preferred': 'Family', 'Point of Contact': 'Contact Agent'},
]
rows = []
for listing in listings:
    r = predict_listing(listing, final, schema)
    rows.append({'City': listing['City'], 'Locality': listing['Area Locality'], 'BHK': listing['BHK'],
                 'Size (sq ft)': listing['Size'], 'Predicted rent (Rs / month)': round(r['rent']),
                 'Range across seeds': f"{r['low']:,.0f} – {r['high']:,.0f}"})
pd.DataFrame(rows)

,City,Locality,BHK,Size (sq ft),Predicted rent (Rs / month),Range across seeds
0,Kolkata,Salt Lake City Sector 2,2,900,13590,"13,148 – 14,357"
1,Mumbai,Andheri West,3,1400,114714,"110,773 – 118,598"


## 3. Incomplete or unusual input — defaults and warnings

In [3]:
r = predict_listing({'City': 'delhi', 'BHK': '2', 'Size': '1,100', 'Bathroom': 2, 'Area Locality': 'Somewhere New'},
                    final, schema)
print(f"Predicted rent: Rs {r['rent']:,.0f} / month")
print('Warnings:', *r['warnings'], sep='\n  - ')

Predicted rent: Rs 21,679 / month
Warnings:
  - Furnishing Status not given — assumed 'Semi-Furnished' (the most common value).
  - Area Type not given — assumed 'Super Area' (the most common value).
  - Tenant Preferred not given — assumed 'Bachelors/Family' (the most common value).
  - Point of Contact not given — assumed 'Contact Owner' (the most common value).
  - current_floor not given — assumed 2.
  - total_floors not given — assumed 4.
  - Locality 'Somewhere New' is not in the training data for Delhi — an average locality effect is used (city, size and the other details still count).


## 4. Invalid input — explained, not crashed

In [4]:
try:
    predict_listing({'City': 'Pune', 'BHK': 0, 'Size': -50, 'Bathroom': 'two', 'current_floor': 7, 'total_floors': 3},
                    final, schema)
except InvalidListing as problem:
    print('Cannot predict:', *problem.errors, sep='\n  - ')

Cannot predict:
  - City must be one of: Mumbai, Chennai, Bangalore, Hyderabad, Delhi, Kolkata (got 'Pune').
  - Bathroom must be a number (got 'two').
  - BHK must be at least 1.
  - Size must be greater than 0 sq ft.
  - current_floor (7) is above the building's total_floors (3).
